# Convolutional Neural Networks
## Project: Landmark Classification & Tagging for Social Media

**Part 1 of 3: build a CNN from scratch.**

In this notebook you will:
1. Build a data pipeline (`src/data.py`) and visualize it.
2. Define a CNN from scratch in `src/model.py`.
3. Define loss & optimizer in `src/optimization.py`.
4. Implement training in `src/train.py`.
5. Train and evaluate the model, then export it to TorchScript.


## 0. Set up the environment

In [ ]:
%load_ext autoreload
%autoreload 2

from src.helpers import setup_env
setup_env()


## 1. Data
Open `src/data.py` and complete the `get_data_loaders` and `visualize_one_batch` functions. Then run the tests below.

In [ ]:
!pytest -vv src/data.py --no-header -x


### 1.1 Visualize a batch
Use the data loaders to grab a batch from the training set and display a few examples with their labels.

In [ ]:
%matplotlib inline
from src.data import get_data_loaders, visualize_one_batch

data_loaders = get_data_loaders(batch_size=32)
fig = visualize_one_batch(data_loaders, max_n=5)


**Question 1**: Describe your data preprocessing and augmentation procedure.

**Answer**: All splits use `Resize(256)` to keep aspect ratio while making the shorter side 256 pixels, then a 224×224 crop (random for train, center for valid/test) — 224 is the canonical input size for ImageNet-style CNNs. The training pipeline adds three augmentations between the crop and `ToTensor()`: `RandomHorizontalFlip(p=0.5)` (landmarks look fine mirrored), `RandomRotation(15)` (camera tilt invariance) and `ColorJitter` (different times of day and weather). Finally, all splits normalize by the per-channel mean and std computed once over the training set so the inputs are centered and scaled.

## 2. Model
Open `src/model.py` and implement the `MyModel` class.

**Question 2**: Outline the steps you took to arrive at your final architecture.

**Answer**: I used a VGG-style backbone: 5 sequential blocks of (Conv 3×3 → BN → ReLU → Conv 3×3 → BN → ReLU → MaxPool 2×2). Each block doubles the channel count (3 → 32 → 64 → 128 → 256 → 512) and halves the spatial size (224 → 112 → 56 → 28 → 14 → 7). Two stacked 3×3 convs per block match the receptive field of a 5×5 conv with fewer parameters. BatchNorm stabilizes training and lets me use a higher learning rate. After the feature extractor an `AdaptiveAvgPool2d(1)` produces a 512-dim embedding, which is fed into a small MLP head (`Linear 512→256 → BN → ReLU → Dropout → Linear 256→num_classes`). Dropout in the head is the main regularizer against overfitting on the relatively small landmark dataset.

In [ ]:
!pytest -vv src/model.py --no-header -x


## 3. Loss and optimizer
Complete `src/optimization.py`.

In [ ]:
!pytest -vv src/optimization.py --no-header -x


## 4. Train and validate
Complete `src/train.py`.

In [ ]:
!pytest -vv src/train.py --no-header -x


### 4.1 Putting it all together

In [ ]:
import torch
from src.data import get_data_loaders
from src.model import MyModel
from src.optimization import get_loss, get_optimizer
from src.train import optimize

batch_size = 64
valid_size = 0.2
num_epochs = 35
num_classes = 50
dropout = 0.4
learning_rate = 0.01
opt = 'sgd'
weight_decay = 1e-4

data_loaders = get_data_loaders(batch_size=batch_size, valid_size=valid_size)
model = MyModel(num_classes=num_classes, dropout=dropout)
optimizer = get_optimizer(model, optimizer=opt, learning_rate=learning_rate, momentum=0.9, weight_decay=weight_decay)
loss = get_loss()

optimize(
    data_loaders,
    model,
    optimizer,
    loss,
    n_epochs=num_epochs,
    save_path='checkpoints/best_val_loss.pt',
    interactive_tracking=True,
)


### 4.2 Test the model

In [ ]:
model.load_state_dict(torch.load('checkpoints/best_val_loss.pt'))
from src.train import one_epoch_test
_ = one_epoch_test(data_loaders['test'], model, loss)


## 5. Export with TorchScript
Complete the `Predictor` class in `src/predictor.py` (it must apply `self.transforms`, run the model and apply `softmax(dim=1)`).

In [ ]:
!pytest -vv src/predictor.py --no-header -x


In [ ]:
from src.predictor import Predictor
from src.helpers import compute_mean_and_std

model.load_state_dict(torch.load('checkpoints/best_val_loss.pt'))
mean, std = compute_mean_and_std()
class_names = data_loaders['train'].dataset.classes

predictor = Predictor(model, class_names=class_names, mean=mean, std=std).cpu()
scripted_predictor = torch.jit.script(predictor)
scripted_predictor.save('checkpoints/original_exported.pt')


### 5.1 Reload the exported model and compute the confusion matrix

In [ ]:
import torch
from src.helpers import plot_confusion_matrix

model_reloaded = torch.jit.load('checkpoints/original_exported.pt')

preds, truths = [], []
with torch.no_grad():
    for images, labels in data_loaders['test']:
        # Predictor expects uint8 images so reverse the normalization the
        # test loader applied.
        from torchvision import transforms
        invTrans = transforms.Compose([
            transforms.Normalize(mean=[0., 0., 0.], std=1.0/std),
            transforms.Normalize(mean=-mean, std=[1., 1., 1.]),
        ])
        uint_imgs = (invTrans(images).clamp(0, 1) * 255).to(torch.uint8)
        probs = model_reloaded(uint_imgs)
        preds.append(probs.argmax(dim=1))
        truths.append(labels)
preds = torch.cat(preds)
truths = torch.cat(truths)
fig = plot_confusion_matrix(preds, truths)
